# 5. Integration

Differentiation shifts a grade **up**; integration shifts it **down**. They are the
same operation at `+1` and `-1` of one signed parameter, which is why the
antiderivative is a composite like any other and the round trip is exact.

What this notebook covers:

1. **definite integrals** — `integrate`
2. **the antiderivative as a number** — and the exact round trip
3. **the increment** — `integrate_step`, `eval_taylor`
4. **improper integrals** — what works, and one that does not
5. **ODEs** — `solve_ode`, with a convergence table

In [1]:
import math, warnings
import composite.composite_lib as cl
import composite.composite_extended as ce
from composite import R, ZERO, Composite, sin, cos, exp, ln, sqrt
from composite.backends.config import use_dict
from mpmath import mp, mpf

use_dict(); cl._refresh_constants()
# MAX_ACTIVE_DIMS is left at its default of 60. Raising it removes the
# truncation guard, and an iterative algorithm then grows terms without
# bound -- solve_ode below did not finish 20 steps in 30s at 10**9, and
# finishes 200 steps in 0.86s at the default.
mp.dps = 40
warnings.simplefilter("ignore")
for w in (cl.CancellationWarning, cl.NotConventionalWarning):
    warnings.filterwarnings("ignore", category=w)
print("ready")

ready


## 1. Definite integrals

`integrate(f, a, b)` takes a composite-valued integrand and returns a **float**.
Checked against 40-digit quadrature.

In [2]:
problems = [("x**2            over [0,1]",  lambda x: x * x,          0, 1,       lambda t: t * t),
            ("x**3 - 2x       over [0,2]",  lambda x: x**3 - R(2)*x,  0, 2,       lambda t: t**3 - 2*t),
            ("sin(x)          over [0,pi]", lambda x: sin(x),         0, math.pi, lambda t: mp.sin(t)),
            ("cos(x)*sin(x)   over [0,1]",  lambda x: cos(x)*sin(x),  0, 1,       lambda t: mp.cos(t)*mp.sin(t)),
            ("exp(x)          over [0,1]",  lambda x: exp(x),         0, 1,       lambda t: mp.e ** t),
            ("x*exp(-x)       over [0,3]",  lambda x: x * exp(-x),    0, 3,       lambda t: t * mp.e ** (-t)),
            ("exp(-x**2)      over [0,2]",  lambda x: exp(-(x * x)),  0, 2,       lambda t: mp.e ** (-(t*t))),
            ("sqrt(x)         over [0,1]",  lambda x: sqrt(x),        0, 1,       lambda t: mp.sqrt(t)),
            ("ln(1+x)         over [0,1]",  lambda x: ln(R(1) + x),   0, 1,       lambda t: mp.log(1 + t))]
print("%-30s %-22s %-22s %s" % ("integral", "composite", "40-digit truth", "rel err"))
for label, f, a, b, exact in problems:
    got  = cl.integrate(f, a, b)
    want = float(mp.quad(exact, [mpf(a), mpf(b)]))
    print("%-30s %-22.15g %-22.15g %.1e"
          % (label, got, want, abs(got - want) / max(abs(want), 1.0)))

integral                       composite              40-digit truth         rel err
x**2            over [0,1]     0.333333333333333      0.333333333333333      5.6e-17


x**3 - 2x       over [0,2]     0                      -6.27049716540098e-44  6.3e-44
sin(x)          over [0,pi]    2                      2                      2.2e-15
cos(x)*sin(x)   over [0,1]     0.354036709136786      0.354036709136786      5.6e-17
exp(x)          over [0,1]     1.71828182845905       1.71828182845905       0.0e+00
x*exp(-x)       over [0,3]     0.800851726528544      0.800851726528544      0.0e+00
exp(-x**2)      over [0,2]     0.882081390762422      0.882081390762422      1.1e-16
sqrt(x)         over [0,1]     0.666666666666886      0.666666666666667      2.2e-13
ln(1+x)         over [0,1]     0.386294361119891      0.386294361119891      0.0e+00


### Quotient integrands

Worth its own table, because this is where `integrate` was broken until recently.
The variable of integration rides on a **lane**, which makes its dimensions
vectors, and the backend's division routine assumed scalar dimensions throughout.
`1/(1+x**2)` raised a `TypeError`, and `x/(1+x)` returned the interval **width**
with no error at all — 226% wrong, while the integrand evaluated correctly at every
point.

In [3]:
quotients = [("x/(1+x)        over [0,1]", lambda x: x / (R(1) + x),        0, 1, 1 - math.log(2)),
             ("1/(1+x**2)     over [0,1]", lambda x: R(1) / (R(1) + x * x), 0, 1, math.pi / 4),
             ("1/(2+x)        over [0,1]", lambda x: R(1) / (R(2) + x),     0, 1, math.log(1.5)),
             ("1/(1+x)        over [0,3]", lambda x: R(1) / (R(1) + x),     0, 3, math.log(4))]
print("%-30s %-22s %-22s %s" % ("integral", "composite", "exact", "rel err"))
for label, f, a, b, want in quotients:
    got = cl.integrate(f, a, b)
    print("%-30s %-22.15g %-22.15g %.1e" % (label, got, want, abs(got - want) / abs(want)))

integral                       composite              exact                  rel err
x/(1+x)        over [0,1]      0.306852819440055      0.306852819440055      0.0e+00
1/(1+x**2)     over [0,1]      0.785398163397448      0.785398163397448      1.4e-16
1/(2+x)        over [0,1]      0.405465108108164      0.405465108108164      0.0e+00
1/(1+x)        over [0,3]      1.38629436111989       1.38629436111989       1.6e-16


## 2. The antiderivative is a composite

`antiderivative` shifts every grade down by one and divides by the new order — the
mirror image of `D(1)`. The result is an ordinary composite, so you can keep
computing with it.

In [4]:
jet = (R(2) + ZERO) ** 2                  # the jet of x**2 at x = 2
F   = cl.antiderivative(jet)

print("jet of x**2 at 2   ", jet)
print("its antiderivative ", F)
print()
print("read it back: F' should be the jet we started from")
print("F.D(1)             ", F.D(1))
print("round trip exact?  ", F.D(1).coeffs_dict() == jet.coeffs_dict())

jet of x**2 at 2    <|4|₀ |4|₋₁ |1|₋₂>
its antiderivative  <|0|₀ |4|₋₁ |2|₋₂ |0.333333|₋₃>

read it back: F' should be the jet we started from
F.D(1)              <|4|₀ |4|₋₁ |1|₋₂>
round trip exact?   True


The grade-0 term of `F` is the constant of integration, which `antiderivative`
leaves at 0 by default. Everything else is `x**3/3` read at 2: `F' = 4`,
`F''/2! = 2`, `F'''/3! = 1/3`.

In [5]:
print("F'(2)    =", F.d(1), "  want x**2 at 2 = 4")
print("F''(2)   =", F.d(2), "  want 2x at 2   = 4")
print("F'''(2)  =", F.d(3), "  want 2         = 2")

F'(2)    = 4.0   want x**2 at 2 = 4
F''(2)   = 4.0   want 2x at 2   = 4
F'''(2)  = 2.0   want 2         = 2


## 3. The increment over a step

`integrate_step(dx)` gives the definite integral over `[x, x+dx]` straight from the
jet — no quadrature, no subdivision. It is `eval_taylor` applied to the
antiderivative, which is why `eval_taylor` returns the **increment** rather than the
value: the antiderivative's constant is arbitrary and must cancel.

In [6]:
jet = (R(2) + ZERO) ** 2                  # x**2 at 2
print("%-10s %-24s %-24s %s" % ("dx", "integrate_step", "exact (x**3/3)", "rel err"))
for dx in (0.5, 0.1, 0.01, 0.001):
    got  = jet.integrate_step(dx)
    want = ((2 + dx) ** 3 - 2 ** 3) / 3
    print("%-10g %-24.15g %-24.15g %.1e" % (dx, got, want, abs(got - want) / abs(want)))

dx         integrate_step           exact (x**3/3)           rel err
0.5        2.54166666666667         2.54166666666667         0.0e+00
0.1        0.420333333333333        0.420333333333334        7.9e-16
0.01       0.0402003333333333       0.0402003333333324       2.4e-14
0.001      0.00400200033333333      0.00400200033333261      1.8e-13


In [7]:
print("and the value of a function a step away, from the same jet:")
f = sin(R(1.0) + ZERO)
for dx in (0.3, 0.1, -0.2):
    got  = f.st() + f.eval_taylor(dx)        # st() + INCREMENT = value
    want = math.sin(1.0 + dx)
    print("  sin(1%+.1f) = %-20.15g want %-20.15g err %.1e" % (dx, got, want, abs(got-want)))

and the value of a function a step away, from the same jet:
  sin(1+0.3) = 0.963558185417192    want 0.963558185417193    err 1.0e-15
  sin(1+0.1) = 0.891207360061435    want 0.891207360061435    err 1.1e-16
  sin(1-0.2) = 0.717356090899523    want 0.717356090899523    err 1.1e-16


## 4. Improper integrals

`improper_integral(f, a)` runs to `+infinity` and returns `(Composite, error)`. It
detects the tail behaviour from a composite evaluation rather than being told.

In [8]:
tails = [("exp(-x)        over [0,inf)",   lambda x: exp(-x),        0, 1.0),
         ("x*exp(-x)      over [0,inf)",   lambda x: x * exp(-x),    0, 1.0),
         ("x**2*exp(-x)   over [0,inf)",   lambda x: x * x * exp(-x), 0, 2.0),
         ("exp(-2x)       over [0,inf)",   lambda x: exp(R(-2) * x),  0, 0.5)]
print("%-32s %-20s %-14s %s" % ("integral", "composite", "want", "rel err"))
for label, f, a, want in tails:
    value, err = cl.improper_integral(f, a)
    got = value.st()
    print("%-32s %-20.15g %-14g %.1e" % (label, got, want, abs(got - want) / abs(want)))

integral                         composite            want           rel err
exp(-x)        over [0,inf)      1                    1              0.0e+00
x*exp(-x)      over [0,inf)      1                    1              0.0e+00
x**2*exp(-x)   over [0,inf)      2                    2              4.4e-16
exp(-2x)       over [0,inf)      0.499999999999961    0.5            7.8e-14


### Power-law tails work too

`improper_integral` divides the same way `integrate` does, so the fix in section 6
reaches it: an inverse-power tail is now handled alongside the exponential ones.

In [9]:
power_laws = [("1/(1+x**2)   over [0,inf)", lambda x: R(1) / (R(1) + x * x),            math.pi / 2),
              ("1/(1+x)**2   over [0,inf)", lambda x: R(1) / ((R(1) + x) * (R(1) + x)), 1.0),
              ("1/(1+x**3)   over [0,inf)", lambda x: R(1) / (R(1) + x ** 3),
               2 * math.pi / (3 * math.sqrt(3)))]
print("%-30s %-20s %-20s %s" % ("integral", "composite", "want", "rel err"))
for label, f, want in power_laws:
    value, _ = cl.improper_integral(f, 0)
    got = value.st()
    print("%-30s %-20.12g %-20.12g %.1e" % (label, got, want, abs(got - want) / abs(want)))
print()
print("to ~1e-9, which is the tail cutoff rather than the arithmetic.")

integral                       composite            want                 rel err
1/(1+x**2)   over [0,inf)      1.57079632743        1.57079632679        4.0e-10
1/(1+x)**2   over [0,inf)      1.00000000238        1                    2.4e-09
1/(1+x**3)   over [0,inf)      1.20919957636        1.20919957616        1.7e-10

to ~1e-9, which is the tail cutoff rather than the arithmetic.


## 5. Ordinary differential equations

`solve_ode(f, (a,b), y0, steps)` returns the solution as a list of `(x, y)` pairs.
The step uses the composite jet of the integrand rather than a fixed Runge-Kutta
tableau, so the order comes from how many grades you keep.

In [10]:
print("y' = y,  y(0) = 1  over [0,1]   ->  e = %.15g" % math.e)
print()
print("%-10s %-24s %s" % ("steps", "y(1)", "rel err"))
for steps in (25, 50, 200, 800):
    points = ce.solve_ode(lambda x, y: y, (0, 1), 1.0, steps=steps)
    got = points[-1][1]
    print("%-10d %-24.15g %.1e" % (steps, got, abs(got - math.e) / math.e))

y' = y,  y(0) = 1  over [0,1]   ->  e = 2.71828182845905

steps      y(1)                     rel err
25         2.71828177236928         2.1e-08
50         2.71828182489456         1.3e-09
200        2.71828182844495         5.2e-12
800        2.718281828459           1.8e-14


In [11]:
print("a nonlinear one: y' = -2xy**2,  y(0) = 1,  exact y = 1/(1+x**2)")
print()
print("%-8s %-22s %-22s %s" % ("x", "solve_ode", "exact", "abs err"))
points = ce.solve_ode(lambda x, y: R(-2) * x * y * y, (0, 2), 1.0, steps=400)
for index in (0, 100, 200, 300, 400):
    x_at, y_at = points[index]
    want = 1.0 / (1.0 + x_at * x_at)
    print("%-8.2f %-22.15g %-22.15g %.1e" % (x_at, y_at, want, abs(y_at - want)))

a nonlinear one: y' = -2xy**2,  y(0) = 1,  exact y = 1/(1+x**2)

x        solve_ode              exact                  abs err
0.00     1                      1                      0.0e+00
0.50     0.799999999996638      0.8                    3.4e-12
1.00     0.500000000004318      0.5                    4.3e-12
1.50     0.307692307697998      0.30769230769231       5.7e-12
2.00     0.200000000003858      0.200000000000003      3.9e-12


## Why the round trip is exact

Differentiation and integration are the same grade shift in opposite directions, so
nothing is approximated when you compose them. The library states it as the reason
fractional calculus falls out: **differentiation and integration are one operation at
`+1` and `-1` of a signed parameter**, and a non-integer step is the fractional
derivative used for viscoelasticity and anomalous diffusion.

In [12]:
start = sin(R(0.7) + ZERO) * exp(R(0.7) + ZERO)
there_and_back = cl.antiderivative(start).D(1)
print("start          ", str(start)[:58])
print("D(antideriv)   ", str(there_and_back)[:58])
worst = max(abs(there_and_back.coeffs_dict().get(g, 0.0) - v)
            for g, v in start.coeffs_dict().items())
print()
print("worst coefficient difference over the whole jet: %.1e" % worst)

start           <|1.2973|₀ |2.8375|₋₁ |1.5402|₋₂ |0.0809693|₋₃ |-0.216216|
D(antideriv)    <|1.2973|₀ |2.8375|₋₁ |1.5402|₋₂ |0.0809693|₋₃ |-0.216216|

worst coefficient difference over the whole jet: 1.2e-32


Two things to carry away. `eval_taylor` is the **increment**, not the value — add
`st()` yourself. And raising `MAX_ACTIVE_DIMS` is a footgun here: it removes the
truncation guard, and `solve_ode` then grows terms without bound, so the default of
60 is what these cells use.